Project Written by Ben 

# Home Security System
Automated system to monitor the home security. The system handles the following:

**Security Alerts:**
- Motion detected when house is in "away" mode
- Door opened when house is in "away" mode
- Multiple sensors triggered simultaneously (possible break-in)

**Safety Alerts:**
- Temperature below 35°F (frozen pipe risk)
- Temperature above 95°F (equipment failure)
- Smoke detected (fire risk)

**Comfort Notifications:**
- Temperature outside comfort range (65-75°F) when home is in "home" mode
- Unusual patterns (like a door left open for >5 minutes)

## 1. Setup
Imports and the thresholds used by the rules.

In [186]:

import random
from datetime import datetime, timedelta
from enum import Enum

TEMP_LOW_F = 35         
TEMP_HIGH_F = 95       
TEMP_COMFORT_LOW_F = 65
TEMP_COMFORT_HIGH_F = 75
DOOR_OPEN_LIMIT = timedelta(minutes=5)  

# What every sensor reads when the system starts (a calm house)
START_MOTION = False      # no activity
START_DOOR = "CLOSED"
START_TEMP_F = 68
START_SMOKE = False       # clear

class Mode(Enum):
    HOME = "HOME"
    AWAY = "AWAY"
    SLEEP = "SLEEP"

class SensorType(Enum):
    MOTION = "motion"
    DOOR = "door"
    TEMPERATURE = "temperature"
    SMOKE = "smoke"

class Location(Enum):
    LIVING_ROOM = "Living Room"
    FRONT = "Front"
    KITCHEN = "Kitchen"
    BEDROOM = "Bedroom"

class AlertLevel(Enum):
    CRITICAL = "CRITICAL"
    HIGH = "HIGH"
    INFO = "INFO"
    

class AlertCategory(Enum):
    SECURITY = "SECURITY"
    SAFETY = "SAFETY"
    COMFORT = "COMFORT"


Using enums to define the mode, sensor types, locations, alert levels and alert categories.

## 2. Data structures
A sensor has a location, a type and a current value. Sensors are built in step 5 as a `Sensor` class.

## 3. If/else logic
The alert rules.

In [187]:
def create_alert(time, level, category, message):
    return {"time": time, "level": level, "category": category, "message": message}

def check_sensor(sensor, mode, now):
    kind = sensor.type
    value = sensor.current_value
    # Name used in messages, e.g. "Front Door" or "Kitchen Temperature"
    name = f"{sensor.location.value} {kind.value.capitalize()}"

    # Safety: every mode
    # If this is a smoke sensor and it detects smoke -> critical safety alert (fire risk)
    if kind == SensorType.SMOKE and value:
        return create_alert(now, AlertLevel.CRITICAL, AlertCategory.SAFETY, f"{name} detected - fire risk!")
    # If this is a temperature sensor and it is below 35°F -> critical safety alert (frozen pipes)
    if kind == SensorType.TEMPERATURE and value < TEMP_LOW_F:
        return create_alert(now, AlertLevel.CRITICAL, AlertCategory.SAFETY, f"{name} at {value}°F - frozen pipe risk!")
    # If this is a temperature sensor and it is above 95°F -> critical safety alert (equipment failure)
    if kind == SensorType.TEMPERATURE and value > TEMP_HIGH_F:
        return create_alert(now, AlertLevel.CRITICAL, AlertCategory.SAFETY, f"{name} at {value}°F - equipment failure risk!")

    # Security: AWAY only
    # If nobody is home (AWAY) and a motion sensor detects movement -> high security alert
    if mode == Mode.AWAY and kind == SensorType.MOTION and value:
        return create_alert(now, AlertLevel.HIGH, AlertCategory.SECURITY, f"{name} detected while in {mode.value} mode!")
    # If nobody is home (AWAY) and a door is open -> high security alert
    if mode == Mode.AWAY and kind == SensorType.DOOR and value == "OPEN":
        return create_alert(now, AlertLevel.HIGH, AlertCategory.SECURITY, f"{name} opened while in {mode.value} mode!")

    # Comfort: HOME only
    # If the family is home and it is a temperature sensor...
    if mode == Mode.HOME and kind == SensorType.TEMPERATURE:
        # ...and it is below 65°F or above 75°F -> comfort notice
        if value < TEMP_COMFORT_LOW_F or value > TEMP_COMFORT_HIGH_F:
            return create_alert(now, AlertLevel.INFO, AlertCategory.COMFORT, f"{name} at {value}°F is outside the {TEMP_COMFORT_LOW_F}-{TEMP_COMFORT_HIGH_F}°F comfort range")

    # None of the rules matched -> everything is normal, no alert
    return None


## 4. Functions
All functions to generate sensor readings, process them, trigger alerts, and log events.

### 4.1 Sensor readings

In [188]:
def generate_reading(sensor):
    kind = sensor.type

    # Motion: about a 15% chance of detecting movement on any reading
    if kind == SensorType.MOTION:
        return random.random() < 0.15

    # Door: about a 10% chance that it is open
    if kind == SensorType.DOOR:
        return "OPEN" if random.random() < 0.10 else "CLOSED"

    # Smoke: rare, about a 2% chance
    if kind == SensorType.SMOKE:
        return random.random() < 0.02

    # Temperature: usually near 68°F, rarely far away from it
    if kind == SensorType.TEMPERATURE:
        return round(random.gauss(68, 12))

### 4.2 Process Readings

In [189]:
def process_reading(sensor, mode, now):
    # Take a new reading (stored inside the sensor)
    sensor.read()
    # Run the rules on it -> an alert, or None
    return sensor.isAbnormal(mode, now)

### 4.3 Trigger and display alters

In [190]:
def show_alert(alert):
    # Print the alert in the same format as the lab's sample output
    print(f"[ALERT!] 🚨 {alert['level'].value}: {alert['category'].value}: {alert['message']}")

In [191]:
show_alert(create_alert(datetime.now(), AlertLevel.HIGH, AlertCategory.SECURITY, "There's a door alert at the Front"))

[ALERT!] 🚨 HIGH: SECURITY: There's a door alert at the Front


### 4.4 Loggin functions

In [192]:
def log_event(time, message):
    # Print the event in the sample's [LOG] format
    print(f"[LOG] [{time.strftime('%H:%M:%S')}] {message}")

In [193]:
log_event(datetime.now(), "Sending notification to homeowner...")

[LOG] [16:36:15] Sending notification to homeowner...


## 5 Creating Classes 

In [194]:
class Sensor:
    def __init__(self, location, sensor_type, value):
        self.location = location
        self.type = sensor_type
        self.current_value = value
        self.start_value = value  # remembered so reset() can go back to it

    def read(self):
        # Take a new reading and store it
        self.current_value = generate_reading(self)

    def isAbnormal(self, mode, now):
        # Run the alert rules -> an alert, or None when everything is normal
        return check_sensor(self, mode, now)

    def reset(self):
        # Go back to the value the sensor started with
        self.current_value = self.start_value

def init_sensors():
    return [
        Sensor(Location.LIVING_ROOM, SensorType.MOTION, START_MOTION),
        Sensor(Location.FRONT, SensorType.DOOR, START_DOOR),
        Sensor(Location.KITCHEN, SensorType.TEMPERATURE, START_TEMP_F),
        Sensor(Location.BEDROOM, SensorType.SMOKE, START_SMOKE),
    ]

In [195]:
sensors = init_sensors()
for s in sensors:
    print("Reading", s.type.value, "sensor in", s.location.value, ":", s.current_value)
    print("-> Alert:", process_reading(s, Mode.AWAY, datetime.now()))
    s.reset()
    print("-> After reset:", s.current_value)

Reading motion sensor in Living Room : False
-> Alert: None
-> After reset: False
Reading door sensor in Front : CLOSED
-> Alert: None
-> After reset: CLOSED
Reading temperature sensor in Kitchen : 68
-> Alert: None
-> After reset: 68
Reading smoke sensor in Bedroom : False
-> Alert: None
-> After reset: False


## 6. Integration
Put everything together: print each sensor's reading, raise alerts, and run it tick by tick.

In [196]:
def describe_reading(sensor):
    # Text for one sensor, e.g. "Front Door: CLOSED" or "Kitchen Temperature: 68°F (Normal)"
    name = f"{sensor.location.value} {sensor.type.value.capitalize()}"
    value = sensor.current_value
    if sensor.type == SensorType.MOTION:
        text = "Motion detected" if value else "No activity"
    elif sensor.type == SensorType.DOOR:
        text = "OPENED" if value == "OPEN" else "CLOSED"
    elif sensor.type == SensorType.SMOKE:
        text = "SMOKE DETECTED" if value else "CLEAR"
    else:
        note = "Normal" if TEMP_COMFORT_LOW_F <= value <= TEMP_COMFORT_HIGH_F else "Outside comfort range"
        text = f"{value}°F ({note})"
    return f"[READING] {name}: {text}"

In [197]:
def raise_alert(alert):
    # Show the alert, then log that the homeowner was notified
    show_alert(alert)
    log_event(alert["time"], "Sending notification to homeowner...")

def run_simulation(mode, start_time, ticks, script=None):
    # script (optional): one dict per tick with fixed sensor values, used by the tests.
    # Without a script, every reading is random.
    sensors = init_sensors()
    now = start_time
    door_open_since = None      # when the door was opened (None = closed)
    door_notice_sent = False    # so one opening only gives one notice

    # Header, then the sensors' starting values
    print("=== HomeGuard Security System ===")
    print(f"Time: {now.strftime('%H:%M:%S')}")
    print(f"Mode: {mode.value}\n")
    for sensor in sensors:
        print(describe_reading(sensor))

    # One tick = one minute: read every sensor, show it, raise any alert
    for tick in range(ticks):
        now += timedelta(minutes=1)
        print(f"\nTime: {now.strftime('%H:%M:%S')}")
        security_alerts = 0

        for sensor in sensors:
            if script:
                # Test mode: use the scripted value, or keep the current one
                sensor.current_value = script[tick].get(sensor.type, sensor.current_value)
                alert = sensor.isAbnormal(mode, now)
            else:
                alert = process_reading(sensor, mode, now)
            print(describe_reading(sensor))
            if alert:
                raise_alert(alert)
                if alert["category"] == AlertCategory.SECURITY:
                    security_alerts += 1

            # Unusual pattern: a door left open for more than 5 minutes
            if sensor.type == SensorType.DOOR:
                if sensor.current_value == "OPEN":
                    if door_open_since is None:
                        door_open_since = now
                    elif now - door_open_since > DOOR_OPEN_LIMIT and not door_notice_sent:
                        minutes = int((now - door_open_since).total_seconds() // 60)
                        raise_alert(create_alert(now, AlertLevel.INFO, AlertCategory.COMFORT,
                                                 f"Front Door has been open for {minutes} minutes"))
                        door_notice_sent = True
                else:
                    door_open_since = None
                    door_notice_sent = False

        # Two or more security alerts in the same tick: possible break-in
        if security_alerts >= 2:
            raise_alert(create_alert(now, AlertLevel.CRITICAL, AlertCategory.SECURITY,
                                     "Multiple sensors triggered - possible break-in!"))

In [198]:
start = datetime.now().replace(hour=14, minute=30, second=0, microsecond=0)
run_simulation(Mode.AWAY, start, 10)

=== HomeGuard Security System ===
Time: 14:30:00
Mode: AWAY

[READING] Living Room Motion: No activity
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 68°F (Normal)
[READING] Bedroom Smoke: CLEAR

Time: 14:31:00
[READING] Living Room Motion: No activity
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 61°F (Outside comfort range)
[READING] Bedroom Smoke: CLEAR

Time: 14:32:00
[READING] Living Room Motion: No activity
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 86°F (Outside comfort range)
[READING] Bedroom Smoke: CLEAR

Time: 14:33:00
[READING] Living Room Motion: No activity
[READING] Front Door: OPENED
[ALERT!] 🚨 HIGH: SECURITY: Front Door opened while in AWAY mode!
[LOG] [14:33:00] Sending notification to homeowner...
[READING] Kitchen Temperature: 58°F (Outside comfort range)
[READING] Bedroom Smoke: CLEAR

Time: 14:34:00
[READING] Living Room Motion: No activity
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 68°F (Normal)
[READIN

## 7. Testing
Fixed test cases
1. AWAY: door opens, then motion
2. AWAY: door + motion in the same tick (possible break-in)
3. SLEEP: temperature boundaries 35/95 and smoke
4. HOME: comfort notice, safety overrides comfort, door open >5 minutes

### Case 1: Security test. AWAY, the door opens, then motion

In [199]:
t = datetime.now().replace(hour=14, minute=30, second=0, microsecond=0)
run_simulation(Mode.AWAY, t, 3, script=[
    {SensorType.DOOR: "OPEN"},
    {SensorType.DOOR: "CLOSED", SensorType.MOTION: True},
    {SensorType.MOTION: False},
])

=== HomeGuard Security System ===
Time: 14:30:00
Mode: AWAY

[READING] Living Room Motion: No activity
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 68°F (Normal)
[READING] Bedroom Smoke: CLEAR

Time: 14:31:00
[READING] Living Room Motion: No activity
[READING] Front Door: OPENED
[ALERT!] 🚨 HIGH: SECURITY: Front Door opened while in AWAY mode!
[LOG] [14:31:00] Sending notification to homeowner...
[READING] Kitchen Temperature: 68°F (Normal)
[READING] Bedroom Smoke: CLEAR

Time: 14:32:00
[READING] Living Room Motion: Motion detected
[ALERT!] 🚨 HIGH: SECURITY: Living Room Motion detected while in AWAY mode!
[LOG] [14:32:00] Sending notification to homeowner...
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 68°F (Normal)
[READING] Bedroom Smoke: CLEAR

Time: 14:33:00
[READING] Living Room Motion: No activity
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 68°F (Normal)
[READING] Bedroom Smoke: CLEAR


### Case 2: Break-in. AWAY, door and motion in the same tick

In [200]:
t = datetime.now().replace(hour=2, minute=10, second=0, microsecond=0)
run_simulation(Mode.AWAY, t, 1, script=[
    {SensorType.DOOR: "OPEN", SensorType.MOTION: True},
])

=== HomeGuard Security System ===
Time: 02:10:00
Mode: AWAY

[READING] Living Room Motion: No activity
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 68°F (Normal)
[READING] Bedroom Smoke: CLEAR

Time: 02:11:00
[READING] Living Room Motion: Motion detected
[ALERT!] 🚨 HIGH: SECURITY: Living Room Motion detected while in AWAY mode!
[LOG] [02:11:00] Sending notification to homeowner...
[READING] Front Door: OPENED
[ALERT!] 🚨 HIGH: SECURITY: Front Door opened while in AWAY mode!
[LOG] [02:11:00] Sending notification to homeowner...
[READING] Kitchen Temperature: 68°F (Normal)
[READING] Bedroom Smoke: CLEAR
[ALERT!] 🚨 CRITICAL: SECURITY: Multiple sensors triggered - possible break-in!
[LOG] [02:11:00] Sending notification to homeowner...


### Case 3: Safety test. SLEEP, 35 and 95 are fine, 34 and 96 are not, then smoke

In [201]:
t = datetime.now().replace(hour=3, minute=0, second=0, microsecond=0)
run_simulation(Mode.SLEEP, t, 6, script=[
    {SensorType.TEMPERATURE: 35},
    {SensorType.TEMPERATURE: 34},
    {SensorType.TEMPERATURE: 95},
    {SensorType.TEMPERATURE: 96},
    {SensorType.TEMPERATURE: 68, SensorType.SMOKE: True},
    {SensorType.SMOKE: False},
])

=== HomeGuard Security System ===
Time: 03:00:00
Mode: SLEEP

[READING] Living Room Motion: No activity
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 68°F (Normal)
[READING] Bedroom Smoke: CLEAR

Time: 03:01:00
[READING] Living Room Motion: No activity
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 35°F (Outside comfort range)
[READING] Bedroom Smoke: CLEAR

Time: 03:02:00
[READING] Living Room Motion: No activity
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 34°F (Outside comfort range)
[ALERT!] 🚨 CRITICAL: SAFETY: Kitchen Temperature at 34°F - frozen pipe risk!
[LOG] [03:02:00] Sending notification to homeowner...
[READING] Bedroom Smoke: CLEAR

Time: 03:03:00
[READING] Living Room Motion: No activity
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 95°F (Outside comfort range)
[READING] Bedroom Smoke: CLEAR

Time: 03:04:00
[READING] Living Room Motion: No activity
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 96°F (Ou

### Case 4: Comfort test. HOME, 62 and 65, 30 (safety wins), then a door left open

In [202]:
t = datetime.now().replace(hour=10, minute=0, second=0, microsecond=0)
run_simulation(Mode.HOME, t, 11, script=[
    {SensorType.TEMPERATURE: 62},
    {SensorType.TEMPERATURE: 65},
    {SensorType.TEMPERATURE: 30},
    {SensorType.TEMPERATURE: 70, SensorType.DOOR: "OPEN"},   # opens at 10:04
] + [{}] * 7)                                                # still open until 10:11

=== HomeGuard Security System ===
Time: 10:00:00
Mode: HOME

[READING] Living Room Motion: No activity
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 68°F (Normal)
[READING] Bedroom Smoke: CLEAR

Time: 10:01:00
[READING] Living Room Motion: No activity
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 62°F (Outside comfort range)
[ALERT!] 🚨 INFO: COMFORT: Kitchen Temperature at 62°F is outside the 65-75°F comfort range
[LOG] [10:01:00] Sending notification to homeowner...
[READING] Bedroom Smoke: CLEAR

Time: 10:02:00
[READING] Living Room Motion: No activity
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 65°F (Normal)
[READING] Bedroom Smoke: CLEAR

Time: 10:03:00
[READING] Living Room Motion: No activity
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 30°F (Outside comfort range)
[ALERT!] 🚨 CRITICAL: SAFETY: Kitchen Temperature at 30°F - frozen pipe risk!
[LOG] [10:03:00] Sending notification to homeowner...
[READING] Bedroom Smoke: CLE